# LoRA SFT on Colab

Run the cells top to bottom. Before starting: **Runtime -> Change runtime type -> GPU**.

You need `lahja-colab.tgz`, made on your Mac with:
```bash
COPYFILE_DISABLE=1 tar czf ~/Desktop/lahja-colab.tgz --exclude '__pycache__' --exclude '._*' \
    src pyproject.toml README.md configs data/sft \
    data/processed/egy_test.jsonl data/processed/massive_ar_test.jsonl
```
`README.md` must be included - `pyproject.toml` references it, and `pip install -e .` fails
without it. Use `tar`, not Finder's "Compress", which adds AppleDouble files (`._sft_lora.py`).

In [ ]:
# 1. Which GPU did we get? T4 -> use Qwen3-1.7B. L4/A100 -> Qwen3-4B.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Upload lahja-colab.tgz (or drag it into the file pane on the left instead)
from google.colab import files

files.upload()

In [ ]:
# 3. Extract and enter the project
!mkdir -p /content/lahja && tar xzf lahja-colab.tgz -C /content/lahja
%cd /content/lahja
!ls data/sft/*

In [ ]:
# 4. Install. Do NOT install the `gpu` extra here - it would replace Colab's working torch.
!pip install -q -e .
!pip install -q -U "trl>=1.13" peft datasets accelerate
# Colab ships torchao 0.10.0; current peft refuses anything below 0.16 even though we never
# use it. We don't need it, so remove it (harmless if it isn't installed).
!pip uninstall -q -y torchao
import torch

print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

## Train

Set `MODEL` to match your GPU, then run both mixes. Defaults match the Mac track:
1,200 steps x batch 16, LoRA rank 16, loss on the assistant JSON only.

**Watch the first 100 steps**: loss should fall below ~0.5 with no jump above 1.0.
If it spikes, stop and rerun with `--lr 1e-4`.

In [ ]:
import os

# Environment variables work in notebook cells AND in a terminal ($MODEL), unlike {MODEL},
# which only expands inside an IPython cell.
os.environ["MODEL"] = "Qwen/Qwen3-4B"  # free T4 / Kaggle: "Qwen/Qwen3-1.7B"
os.environ["TAG"] = "qwen3-4b"  # short name used in adapter/result folders
print(os.environ["MODEL"], os.environ["TAG"])

In [ ]:
# D = Saudi/MSA + Egyptian synthetic
!python -m lahja.train.sft_lora --model $MODEL --mix D --out adapters/D-$TAG

In [ ]:
# C = Saudi/MSA only (the comparison run)
!python -m lahja.train.sft_lora --model $MODEL --mix C --out adapters/C-$TAG

In [ ]:
# 5. Evaluate both with the same code used everywhere else
!python -m lahja.eval.run_eval --backend hf --model $MODEL --adapter adapters/D-$TAG --prompt sft --run-name D-$TAG --eval-set data/processed/egy_test.jsonl --eval-set data/processed/massive_ar_test.jsonl --limit 300
!python -m lahja.eval.run_eval --backend hf --model $MODEL --adapter adapters/C-$TAG --prompt sft --run-name C-$TAG --eval-set data/processed/egy_test.jsonl --eval-set data/processed/massive_ar_test.jsonl --limit 300

In [ ]:
# 6. Download results + adapters BEFORE the session dies
!tar czf /content/lahja-results.tgz results adapters
from google.colab import files

files.download("/content/lahja-results.tgz")

On the Mac, from the project root: `tar xzf ~/Downloads/lahja-results.tgz`

Results are tagged per model, so nothing overwrites the 0.6B runs.